In [1]:
import pandas as pd
import re
import nltk
import spacy
from nltk.corpus import stopwords
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib


In [2]:
# Загрузка ресурсов NLTK

resources = ["punkt", "stopwords"]

for r in resources:
    nltk.download(r)

stop_words = set(stopwords.words("english"))


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Валерия\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Валерия\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
# Загрузка ресурсов NLTK

resources = ["punkt", "stopwords"]

for r in resources:
    nltk.download(r)

stop_words = set(stopwords.words("english"))



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Валерия\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Валерия\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])


In [5]:
# Загрузка датасета

train = pd.read_csv("data/amazon_review_polarity_csv/train.csv", header=None)
train.columns = ["polarity", "title", "text"]

test = pd.read_csv("data/amazon_review_polarity_csv/test.csv", header=None)
test.columns = ["polarity", "title", "text"]

In [6]:
# Преобразование меток

train["label"] = train["polarity"].map({1:0, 2:1})
test["label"] = test["polarity"].map({1:0, 2:1})

In [7]:
# Объединение заголовка и текста

train["full_text"] = train["title"] + " " + train["text"]
test["full_text"] = test["title"] + " " + test["text"]


In [8]:
# Для MVP берем часть данных

train = train.sample(20000, random_state=42)
test = test.sample(5000, random_state=42)

In [9]:
def clean_text(text):
    """Очистка текста"""
    # если NaN → пустая строка
    if pd.isna(text):
        text = ""
    else:
        text = str(text)

    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\d+", "", text)

    return text

In [10]:
train["clean_temp"] = train["full_text"].apply(clean_text)
test["clean_temp"] = test["full_text"].apply(clean_text)


In [11]:

# Быстрая лемматизация через spaCy

def preprocess_texts(texts):

    processed_texts = []

    for doc in tqdm(nlp.pipe(texts, batch_size=1000), total=len(texts)):

        tokens = [
            token.lemma_
            for token in doc
            if token.is_alpha
            and token.lemma_ not in stop_words
        ]

        processed_texts.append(" ".join(tokens))

    return processed_texts

In [12]:
train["clean_text"] = preprocess_texts(train["clean_temp"].tolist())
test["clean_text"] = preprocess_texts(test["clean_temp"].tolist())

100%|██████████| 5000/5000 [00:26<00:00, 185.86it/s]


In [13]:

# Удаляем временную колонку

train = train.drop(columns=["clean_temp"])
test = test.drop(columns=["clean_temp"])


In [14]:
# Сохранение результата

train.to_csv("train_clean.csv", index=False)
test.to_csv("test_clean.csv", index=False)

print(train.head())

         polarity                                   title  \
2079998         1                          Expensive Junk   
1443106         1                          Toast too dark   
3463669         2   Excellent imagery...dumbed down story   
2914699         1  Are we pretending everyone is married?   
1603231         1                     Not worth your time   

                                                      text  label  \
2079998  This product consists of a piece of thin flexi...      0   
1443106  Even on the lowest setting, the toast is too d...      0   
3463669  I enjoyed this disc. The video is stunning. I ...      1   
2914699  The authors pretend that parents neither die n...      0   
1603231  Might as well just use a knife, this product h...      0   

                                                 full_text  \
2079998  Expensive Junk This product consists of a piec...   
1443106  Toast too dark Even on the lowest setting, the...   
3463669  Excellent imagery...dum

In [15]:

# Разделение данных
X = train["clean_text"]
y = train["label"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [16]:
# TF-IDF

vectorizer = TfidfVectorizer(max_features=10000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)


In [17]:
# Logistic Regression
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [18]:

# Оценка модели
y_pred = clf.predict(X_val_tfidf)
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.86      0.86      0.86      1974
           1       0.86      0.87      0.86      2026

    accuracy                           0.86      4000
   macro avg       0.86      0.86      0.86      4000
weighted avg       0.86      0.86      0.86      4000



In [19]:
# Сохраняем модель и векторизатор
joblib.dump(clf, "backend/models/logreg_model.pkl")
joblib.dump(vectorizer, "backend/models/tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']

In [27]:
clf = joblib.load("backend/models/logreg_model.pkl")
vectorizer = joblib.load("backend/models/tfidf_vectorizer.pkl")

def predict_emotion(text):
    # Простейшая очистка текста перед TF-IDF
    import re
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    # Преобразуем через TF-IDF
    X_tfidf = vectorizer.transform([text])
    pred = clf.predict(X_tfidf)[0]
    return "Positive" if pred else "Negative"

# Пример
print(predict_emotion("I'm normal with this product."))
print(predict_emotion("This is the worst experience ever."))

Negative
Positive
